# Generating the CSV


In [7]:
# Uninstall any old versions
!pip uninstall enka enka-api enka-py -y

# Install the latest enka
!pip install enka
!pip install enkanetwork
# Now update the assets using the client
import enka
!pip install enkanetwork.py

async def update_assets():
    async with enka.GenshinClient() as client:
        await client.update_assets()
        print("Assets updated successfully!")

await update_assets()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.0/70.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.0 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement enkanetwork (from versions: none)
ERROR: No matching distribution found for enkanetwork
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.0/423.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for enkanetwork.py: filename=enkanetwork.py-1.4.5-py3-none-any.whl size=431813 sha256=32ee8fe8c072116fbf38d584da814ce7af58a26720e42b4deaa3895601031e71
  Stored in directory: /root/.cache/pip/wheels/8f/0c/d1/cebfa55fb8222d1997ff3f51dca8d2857bdcf2a31a7093e03c
Successfully built enkanetwork.py
Assets updated successfully!


In [8]:
import requests

def fetch_player_data(uid):
    response = requests.get(f'https://enka.network/api/uid/{uid}/')
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching data: {response.status_code}")
        return None

def count_constellations(player_data):
    # The 'talentIdList' holds the IDs of unlocked constellations.
    # A player's constellation level is the length of this list.
    if player_data and 'avatarInfoList' in player_data:
        for character in player_data['avatarInfoList']:
            # This gets the list of unlocked constellation IDs.
            talent_ids = character.get('talentIdList', [])
            const_level = len(talent_ids)
            print(f"Character {character.get('name')} has constellation level: {const_level}")
    else:
        print("No character data found.")

# Example usage with the UID from your error
uid = 618867267
player_data = fetch_player_data(uid)
count_constellations(player_data)


Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 3
Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 0
Character None has constellation level: 5
Character None has constellation level: 5


In [9]:
import enka
import pprint

UID = 618867267

async def diagnose():
    async with enka.GenshinClient(enka.gi.Language.ENGLISH) as client:
        response = await client.fetch_showcase(UID)
        char = response.characters[0]  # first character, e.g., Ganyu
        print(f"Character: {char.name}")
        print("\n--- Attributes of 'char' ---")
        print(dir(char))
        print("\n--- char.stats ---")
        for key, stat in char.stats.items():
            print(f"  {key}: value={stat.value}, formatted={stat.formatted_value}")
        print("\n--- char.base_stats (if exists) ---")
        if hasattr(char, 'base_stats'):
            print(char.base_stats)
        else:
            print("No 'base_stats' attribute")
        print("\n--- char.character_data (if exists) ---")
        if hasattr(char, 'character_data'):
            print("character_data has:", dir(char.character_data))
            if hasattr(char.character_data, 'base_stats'):
                print("  base_stats:", char.character_data.base_stats)
        else:
            print("No 'character_data' attribute")
        print("\n--- Weapon ---")
        weapon = char.weapon
        if weapon:
            print(f"  Name: {weapon.name}")
            print("  Attributes:", dir(weapon))
            print("  weapon.stats:", weapon.stats if hasattr(weapon, 'stats') else "None")
            if hasattr(weapon, 'stats') and weapon.stats:
                for stat in weapon.stats:
                    print(f"    {stat.type}: {stat.value}")
            if hasattr(weapon, 'main_stat'):
                print(f"  weapon.main_stat: {weapon.main_stat}")
        else:
            print("No weapon")

await diagnose()

Character: Ganyu

--- Attributes of 'char' ---
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '__pydantic_serializer__'

In [10]:
import enka
import csv
import os
from collections import OrderedDict, Counter

UID = 618867267
CSV_FILENAME = "genshin_data_export.csv"


def get_constellation_count(character):
    if hasattr(character, 'constellations') and character.constellations:
        try:
            if hasattr(character.constellations[0], 'activated'):
                return sum(1 for c in character.constellations if c.activated)
            elif hasattr(character.constellations[0], 'unlocked'):
                return sum(1 for c in character.constellations if c.unlocked)
        except Exception:
            pass
    if hasattr(character, 'constellations_unlocked'):
        try:
            return int(character.constellations_unlocked)
        except Exception:
            pass
    return 0


def artifact_set_bonus(artifact_set_names):
    sets = [name for name in artifact_set_names if name]
    if not sets:
        return "None"
    counter = Counter(sets)
    bonuses = []
    for set_name, count in counter.items():
        if count >= 4:
            bonuses.append(f"{set_name} x4")
        elif count >= 2:
            bonuses.append(f"{set_name} x2")
    return " + ".join(bonuses) if bonuses else "None"


async def main():
    async with enka.GenshinClient(enka.gi.Language.ENGLISH) as client:
        response = await client.fetch_showcase(UID)

        header = [
            "UID", "Player Nickname", "Player Level", "Player Signature",
            "Character Name", "Character Level", "Character Ascension",
            "Character Friendship", "Character Constellations",
            "Weapon Name", "Weapon Level", "Weapon Ascension",
            "Weapon Refinement", "Weapon Stars",
            "Talent NA Level", "Talent Skill Level", "Talent Burst Level",
        ]

        stat_map = {
            enka.gi.FightPropType.FIGHT_PROP_HP: "HP",
            enka.gi.FightPropType.FIGHT_PROP_ATTACK: "ATK",
            enka.gi.FightPropType.FIGHT_PROP_DEFENSE: "DEF",
            enka.gi.FightPropType.FIGHT_PROP_CRITICAL: "Crit Rate",
            enka.gi.FightPropType.FIGHT_PROP_CRITICAL_HURT: "Crit DMG",
            enka.gi.FightPropType.FIGHT_PROP_CHARGE_EFFICIENCY: "Energy Recharge",
            enka.gi.FightPropType.FIGHT_PROP_ELEMENT_MASTERY: "Elemental Mastery",
            enka.gi.FightPropType.FIGHT_PROP_HEAL_ADD: "Healing Bonus",
            enka.gi.FightPropType.FIGHT_PROP_PHYSICAL_ADD_HURT: "Physical DMG Bonus",
            enka.gi.FightPropType.FIGHT_PROP_FIRE_ADD_HURT: "Pyro DMG Bonus",
            enka.gi.FightPropType.FIGHT_PROP_WATER_ADD_HURT: "Hydro DMG Bonus",
            enka.gi.FightPropType.FIGHT_PROP_WIND_ADD_HURT: "Anemo DMG Bonus",
            enka.gi.FightPropType.FIGHT_PROP_ICE_ADD_HURT: "Cryo DMG Bonus",
            enka.gi.FightPropType.FIGHT_PROP_ELEC_ADD_HURT: "Electro DMG Bonus",
            enka.gi.FightPropType.FIGHT_PROP_GRASS_ADD_HURT: "Dendro DMG Bonus",
        }
        for stat_name in stat_map.values():
            header.append(stat_name)

        header.append("Artifact Set Bonus")
        for i in range(5):
            header.extend([
                f"Artifact {i+1} Set", f"Artifact {i+1} Name",
                f"Artifact {i+1} Main Stat Type", f"Artifact {i+1} Main Stat Value",
                f"Artifact {i+1} Sub 1", f"Artifact {i+1} Sub 2",
                f"Artifact {i+1} Sub 3", f"Artifact {i+1} Sub 4",
            ])

        data = OrderedDict()
        if os.path.exists(CSV_FILENAME):
            with open(CSV_FILENAME, "r", encoding="utf-8") as f:
                reader = csv.reader(f)
                existing_header = next(reader, None)
                if existing_header == header:
                    for row in reader:
                        key = (row[0], row[4])
                        data[key] = row
                else:
                    print("Header mismatch – starting fresh.")
        else:
            print("No existing file – creating new.")

        # --- Special numeric keys for final HP/ATK/DEF ---
        # These are the keys that hold the fully calculated totals (base + weapon + artifacts)
        FINAL_STAT_KEYS = {
            enka.gi.FightPropType.FIGHT_PROP_HP: 2000,
            enka.gi.FightPropType.FIGHT_PROP_ATTACK: 2001,
            enka.gi.FightPropType.FIGHT_PROP_DEFENSE: 2002,
        }

        for character in response.characters:
            char_name = character.name
            char_level = character.level
            char_ascension = character.ascension
            char_friendship = character.friendship_level
            char_constellations = get_constellation_count(character)

            weapon = character.weapon
            if weapon:
                weapon_name = weapon.name
                weapon_level = weapon.level
                weapon_ascension = weapon.ascension
                weapon_refinement = weapon.refinement
                weapon_stars = weapon.rarity
            else:
                weapon_name = weapon_level = weapon_ascension = weapon_refinement = weapon_stars = ""

            talents = character.talents
            talent_na = talents[0].level if len(talents) > 0 else ""
            talent_skill = talents[1].level if len(talents) > 1 else ""
            talent_burst = talents[2].level if len(talents) > 2 else ""

            row = [
                str(UID),
                response.player.nickname,
                response.player.level,
                response.player.signature,
                char_name,
                char_level,
                char_ascension,
                char_friendship,
                char_constellations,
                weapon_name,
                weapon_level,
                weapon_ascension,
                weapon_refinement,
                weapon_stars,
                talent_na,
                talent_skill,
                talent_burst,
            ]

            # --- Stats extraction ---
            for prop_type in stat_map.keys():
                if prop_type in (enka.gi.FightPropType.FIGHT_PROP_HP,
                                 enka.gi.FightPropType.FIGHT_PROP_ATTACK,
                                 enka.gi.FightPropType.FIGHT_PROP_DEFENSE):
                    # Use the special numeric keys that hold final totals
                    special_key = FINAL_STAT_KEYS.get(prop_type)
                    stat_obj = character.stats.get(special_key)
                    if stat_obj:
                        val = stat_obj.value
                    else:
                        # fallback to the normal key if special key missing
                        stat_obj = character.stats.get(prop_type)
                        val = stat_obj.value if stat_obj else 0.0
                    row.append(str(val))
                else:
                    stat_obj = character.stats.get(prop_type)
                    row.append(stat_obj.formatted_value if stat_obj else "")

            # --- Artifact processing ---
            artifacts = list(character.artifacts) + [None] * (5 - len(character.artifacts))
            artifact_set_names = []
            artifact_details = []

            for artifact in artifacts:
                if artifact:
                    set_name = artifact.set_name
                    artifact_name = artifact.name
                    main_stat_type = artifact.main_stat.type.name
                    main_stat_value = artifact.main_stat.value
                    subs = []
                    for sub in artifact.sub_stats[:4]:
                        subs.append(f"{sub.type.name}+{sub.value}")
                    subs += [""] * (4 - len(subs))
                else:
                    set_name = artifact_name = ""
                    main_stat_type = main_stat_value = ""
                    subs = ["", "", "", ""]
                artifact_set_names.append(set_name)
                artifact_details.append([set_name, artifact_name, main_stat_type, main_stat_value] + subs)

            set_bonus = artifact_set_bonus(artifact_set_names)
            row.append(set_bonus)

            for details in artifact_details:
                row.extend(details)

            if len(row) < len(header):
                row += [""] * (len(header) - len(row))
            else:
                row = row[:len(header)]

            key = (str(UID), char_name)
            data[key] = row
            print(f"  Processed: {char_name} (C{char_constellations}) – {set_bonus}")

        with open(CSV_FILENAME, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(data.values())

        print(f"\n✅ Data updated for UID {UID}.")
        print(f"   Total characters stored: {len(data)}")
        print(f"   File saved as: {CSV_FILENAME}")

await main()

  Processed: Ganyu (C0) – Blizzard Strayer x4
  Processed: Raiden Shogun (C0) – Emblem of Severed Fate x4
  Processed: Skirk (C0) – Finale of the Deep Galleries x2
  Processed: Neuvillette (C0) – Marechaussee Hunter x4
  Processed: Xiangling (C3) – Emblem of Severed Fate x4
  Processed: Venti (C0) – Viridescent Venerer x4
  Processed: Kinich (C0) – Obsidian Codex x4
  Processed: Rosaria (C0) – Blizzard Strayer x4
  Processed: Nicole (C0) – Celestial Gift x4
  Processed: Durin (C0) – A Day Carved From Rising Winds x4
  Processed: Xingqiu (C5) – Emblem of Severed Fate x4
  Processed: Bennett (C5) – Noblesse Oblige x4

✅ Data updated for UID 618867267.
   Total characters stored: 46
   File saved as: genshin_data_export.csv


# GCSim CLI


In [4]:
# Download the Linux executable directly (no .tar.gz extension)
!wget https://github.com/genshinsim/gcsim/releases/download/v2.42.2/gcsim_linux_amd64

# Make it executable
!chmod +x gcsim_linux_amd64

# (Optional) Rename to just 'gcsim' for convenience
!mv gcsim_linux_amd64 gcsim

--2026-06-18 14:40:58--  https://github.com/genshinsim/gcsim/releases/download/v2.42.2/gcsim_linux_amd64
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/376835082/8f9592f1-48df-4f20-925e-656c7aa67b2e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-18T15%3A31%3A47Z&rscd=attachment%3B+filename%3Dgcsim_linux_amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-18T14%3A31%3A00Z&ske=2026-06-18T15%3A31%3A47Z&sks=b&skv=2018-11-09&sig=NSfCjh5tkaQk7T017qQc5HYPi8etnhYPSTgvK1TROpo%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MTc5NTQ1OCwibmJmIjoxNzgxNzkzNjU4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5i

In [ ]:
!./gcsim -h

Usage of ./gcsim:
  -c string
    	which profile to use (default "config.txt")
  -cpuprofile string
    	write cpu profile to a file. supply file path (otherwise empty string for disabled). 
    	can be viewed in the browser via "go tool pprof -http=localhost:3000 cpu.prof" (insert your desired host/port/filename, requires Graphviz)
  -gz
    	gzip json results; require out flag
  -ks
    	keep serving same results without terminating web server
  -memprofile string
    	write memory profile to a file. supply file path (otherwise empty string for disabled). 
    	can be viewed in the browser via "go tool pprof -http=localhost:3000 mem.prof" (insert your desired host/port/filename, requires Graphviz)
  -nb
    	disable opening default browser
  -nr
    	disable running the simulation (useful if you only want to generate a sample
  -options string
    	Additional options for substat optimization mode. Currently supports the following flags, set in a semi-colon delimited list (e.g. -optio

# GCSim Pareto

In [11]:
import pandas as pd
import subprocess
import json
import random
import re
import tempfile
import os
from datetime import datetime

# -----------------------------------------------------------------
#  Parameterised run function – call this with your custom settings!
# -----------------------------------------------------------------
def run_optimizer(
    lock_chars=None,
    csv_path="genshin_data_export.csv",
    gcsim_bin="./gcsim",
    sim_duration=20,
    sim_iterations=150,
    population_size=100,
    generations=40,
    mutation_rate=0.15,
    crossover_rate=0.8,
    tournament_size=3,
    min_character_level=50,
    traveler_override=None,
    traveler_default="anemo",
    save_prefix="default",
    pareto=True               # Set to False for pure DPS optimisation (no Pareto archive)
):
    # ---- Canonical gcsim data (kept inside to avoid global clutter) ----
    GCSIM_CHARACTER_KEYS = {
        'aino', 'albedo', 'alhaitham', 'aloy', 'amber', 'arlecchino', 'ayaka', 'ayato',
        'baizhu', 'barbara', 'beidou', 'bennett', 'candace', 'charlotte', 'chasca',
        'chevreuse', 'chiori', 'chongyun', 'citlali', 'clorinde', 'collei', 'columbina',
        'cyno', 'dahlia', 'dehya', 'diluc', 'diona', 'dori', 'emilie', 'escoffier', 'eula',
        'faruzan', 'fischl', 'flins', 'freminet', 'furina', 'gaming', 'ganyu', 'gorou',
        'heizou', 'hutao', 'ineffa', 'itto', 'jean', 'kaeya', 'kaveh', 'kazuha', 'keqing',
        'kinich', 'kirara', 'klee', 'kokomi', 'kuki', 'lanyan', 'lauma', 'layla', 'lisa',
        'lynette', 'lyney', 'mavuika', 'mika', 'mizuki', 'mona', 'mualani', 'nahida',
        'navia', 'neuvillette', 'nilou', 'ningguang', 'noelle', 'ororon', 'qiqi', 'raiden',
        'razor', 'rosaria', 'sara', 'sayu', 'sethos', 'shenhe', 'sigewinne', 'skirk',
        'sucrose', 'tartaglia', 'thoma', 'tighnari', 'varesa', 'venti', 'wanderer',
        'wriothesley', 'xiangling', 'xianyun', 'xiao', 'xilonen', 'xingqiu', 'xinyan',
        'yaemiko', 'yanfei', 'yaoyao', 'yelan', 'yoimiya', 'yunjin', 'zhongli',
    }
    TRAVELER_ELEMENTS = ('anemo', 'geo', 'electro', 'dendro', 'hydro', 'pyro', 'cryo')
    for _e in TRAVELER_ELEMENTS:
        GCSIM_CHARACTER_KEYS.add(f'aether{_e}')
        GCSIM_CHARACTER_KEYS.add(f'lumine{_e}')

    CHARACTER_NAME_OVERRIDES = {
        'kamisatoayaka': 'ayaka', 'kamisatoayato': 'ayato', 'raidenshogun': 'raiden',
        'sangonomiyakokomi': 'kokomi', 'kujousara': 'sara', 'kukishinobu': 'kuki',
        'shikanoinheizou': 'heizou', 'aratakiitto': 'itto', 'kaedeharakazuha': 'kazuha',
        'childe': 'tartaglia', 'scaramouche': 'wanderer', 'kusanali': 'nahida',
        'lesserlordkusanali': 'nahida',
    }

    # Full weapon/set/stat dictionaries
    GCSIM_WEAPON_KEYS = {
        'absolution', 'akuoumaru', 'alleyhunter', 'amenomakageuchi', 'amosbow', 'apprenticesnotes',
        'aquasimulacra', 'aquilafavonia', 'ashgravendrinkinghorn', 'astralvulturescrimsonplumage',
        'athousandblazingsuns', 'athousandfloatingdreams', 'azurelight', 'balladoftheboundlessblue',
        'balladofthefjords', 'beaconofthereedsea', 'beginnersprotector', 'blackcliffagate',
        'blackclifflongsword', 'blackcliffpole', 'blackcliffslasher', 'blackcliffwarbow',
        'blackmarrowlantern', 'blacktassel', 'bloodsoakedruins', 'bloodtaintedgreatsword',
        'calamityofeshu', 'calamityqueller', 'cashflowsupervision', 'chainbreaker', 'cinnabarspindle',
        'cloudforged', 'compoundbow', 'coolsteel', 'cranesechoingcall', 'crescentpike',
        'crimsonmoonssemblance', 'darkironsword', 'dawningfrost', 'deathmatch', 'debateclub',
        'dialoguesofthedesertsages', 'dodocotales', 'dragonsbane', 'dragonspinespear', 'dullblade',
        'earthshaker', 'elegyfortheend', 'emeraldorb', 'endoftheline', 'engulfinglightning',
        'etherlightspindlelute', 'everlastingmoonglow', 'eyeofperception', 'fadingtwilight',
        'fangofthemountainking', 'favoniuscodex', 'favoniusgreatsword', 'favoniuslance',
        'favoniussword', 'favoniuswarbow', 'ferrousshadow', 'festeringdesire', 'filletblade',
        'finaleofthedeep', 'fleuvecendreferryman', 'flowerwreathedfeathers', 'flowingpurity',
        'fluteofezpitzal', 'footprintoftherainbow', 'forestregalia', 'fracturedhalo', 'freedomsworn',
        'frostbearer', 'fruitfulhook', 'fruitoffulfillment', 'hakushinring', 'halberd', 'hamayumi',
        'harangeppakufutsu', 'harbingerofdawn', 'huntersbow', 'hunterspath', 'ibispiercer',
        'ironpoint', 'ironsting', 'jadefallssplendor', 'kagotsurubeisshin', 'kagurasverity',
        'katsuragikirinagamasa', 'keyofkhajnisut', 'kingssquire', 'kitaincrossspear',
        'lightoffoliarincision', 'lionsroar', 'lithicblade', 'lithicspear',
        'lostprayertothesacredwinds', 'lumidouceelegy', 'luxurioussealord', 'magicguide',
        'mailedflower', 'makhairaaquamarine', 'mappamare', 'masterkey', 'memoryofdust', 'messenger',
        'missivewindspear', 'mistsplitterreforged', 'mitternachtswaltz', 'moonpiercer',
        'moonweaversdawn', 'mountainbracingbolt', 'mouunsmoon', 'nightweaverslookingglass',
        'nocturnescurtaincall', 'oathsworneye', 'oldmercspal', 'otherworldlystory', 'peakpatrolsong',
        'pocketgrimoire', 'polarstar', 'portablepowersaw', 'predator', 'primordialjadecutter',
        'primordialjadewingedspear', 'prospectorsdrill', 'prospectorsshovel', 'prototypeamber',
        'prototypearchaic', 'prototypecrescent', 'prototyperancour', 'prototypestarglitter',
        'rainbowserpentbow', 'rainslasher', 'rangegauge', 'ravenbow', 'recurvebow',
        'redhornstonethresher', 'reliquaryoftruth', 'rightfulreward', 'ringofyaxche', 'royalbow',
        'royalgreatsword', 'royalgrimoire', 'royallongsword', 'royalspear', 'rust', 'sacrificersstaff',
        'sacrificialbow', 'sacrificialfragments', 'sacrificialgreatsword', 'sacrificialjade',
        'sacrificialsword', 'sapwoodblade', 'scionoftheblazingsun', 'seasonedhuntersbow',
        'sequenceofsolitude', 'serenityscall', 'serpentspine', 'sharpshootersoath',
        'silvershowerheartstrings', 'silversword', 'skyridergreatsword', 'skyridersword',
        'skywardatlas', 'skywardblade', 'skywardharp', 'skywardpride', 'skywardspine', 'slingshot',
        'snarehook', 'snowtombedstarsilver', 'solarpearl', 'songofbrokenpines', 'songofstillness',
        'splendoroftranquilwaters', 'staffofhoma', 'staffofthescarletsands', 'starcallerswatch',
        'sturdybone', 'summitshaper', 'sunnymorningsleepin', 'surfsup', 'swordofdescension',
        'swordofnarzissenkreuz', 'symphonistofscents', 'talkingstick', 'tamayurateinoohanashi',
        'thealleyflash', 'thebell', 'theblacksword', 'thecatch', 'thedockhandsassistant',
        'thefirstgreatmagic', 'theflute', 'thestringless', 'theunforged', 'theviridescenthunt',
        'thewidsith', 'thrillingtalesofdragonslayers', 'thunderingpulse', 'tidalshadow',
        'tomeoftheeternalflow', 'toukaboushigure', 'travelershandysword', 'tulaytullahsremembrance',
        'twinnephrite', 'ultimateoverlordsmegamagicsword', 'urakumisugiri', 'verdict', 'vividnotions',
        'vortexvanquisher', 'wanderingevenstar', 'wastergreatsword', 'wavebreakersfin',
        'waveridingwhirl', 'whiteblind', 'whiteirongreatsword', 'whitetassel', 'windblumeode',
        'wineandsong', 'wolffang', 'wolfsgravestone', 'xiphosmoonlight',
    }
    GCSIM_SET_KEYS = {
        'adventurer', 'archaicpetra', 'aubadeofmorningstarandmoon', 'berserker', 'blizzardstrayer',
        'bloodstainedchivalry', 'braveheart', 'crimsonwitchofflames', 'deepwoodmemories',
        'defenderswill', 'desertpavilionchronicle', 'echoesofanoffering', 'emblemofseveredfate',
        'finaleofthedeepgalleries', 'flowerofparadiselost', 'fragmentofharmonicwhimsy', 'gambler',
        'gildeddreams', 'gladiatorsfinale', 'goldentroupe', 'heartofdepth', 'huskofopulentdreams',
        'instructor', 'lavawalker', 'longnightsoath', 'luckydog', 'maidenbeloved',
        'marechausseehunter', 'martialartist', 'nightoftheskysunveiling',
        'nighttimewhispersintheechoingwoods', 'noblesseoblige', 'nymphsdream', 'obsidiancodex',
        'oceanhuedclam', 'paleflame', 'prayersfordestiny', 'prayersforillumination',
        'prayersforwisdom', 'prayerstospringtime', 'resolutionofsojourner', 'retracingbolide',
        'scholar', 'scrolloftheheroofcindercity', 'shimenawasreminiscence', 'silkenmoonsserenade',
        'songofdayspast', 'tenacityofthemillelith', 'theexile', 'thunderingfury', 'thundersoother',
        'tinymiracle', 'travelingdoctor', 'unfinishedreverie', 'vermillionhereafter',
        'viridescentvenerer', 'vourukashasglow', 'wandererstroupe',
    }
    FIGHT_PROP_TO_GCSIM = {
        'FIGHT_PROP_HP':                ('hp',       False),
        'FIGHT_PROP_HP_PERCENT':        ('hp%',      True),
        'FIGHT_PROP_ATTACK':            ('atk',      False),
        'FIGHT_PROP_ATTACK_PERCENT':    ('atk%',     True),
        'FIGHT_PROP_DEFENSE':           ('def',      False),
        'FIGHT_PROP_DEFENSE_PERCENT':   ('def%',     True),
        'FIGHT_PROP_CRITICAL':          ('cr',       True),
        'FIGHT_PROP_CRITICAL_HURT':     ('cd',       True),
        'FIGHT_PROP_CHARGE_EFFICIENCY': ('er',       True),
        'FIGHT_PROP_ELEMENT_MASTERY':   ('em',       False),
        'FIGHT_PROP_HEAL_ADD':          ('heal',     True),
        'FIGHT_PROP_FIRE_ADD_HURT':     ('pyro%',    True),
        'FIGHT_PROP_WATER_ADD_HURT':    ('hydro%',   True),
        'FIGHT_PROP_WIND_ADD_HURT':     ('anemo%',   True),
        'FIGHT_PROP_ICE_ADD_HURT':      ('cryo%',    True),
        'FIGHT_PROP_ELEC_ADD_HURT':     ('electro%', True),
        'FIGHT_PROP_GRASS_ADD_HURT':    ('dendro%',  True),
        'FIGHT_PROP_ROCK_ADD_HURT':     ('geo%',     True),
        'FIGHT_PROP_PHYSICAL_ADD_HURT': ('phys%',    True),
    }
    ELEMENT_DMG_PROP_TO_ELEMENT = {
        'FIGHT_PROP_FIRE_ADD_HURT':  'pyro',
        'FIGHT_PROP_WATER_ADD_HURT': 'hydro',
        'FIGHT_PROP_WIND_ADD_HURT':  'anemo',
        'FIGHT_PROP_ICE_ADD_HURT':   'cryo',
        'FIGHT_PROP_ELEC_ADD_HURT':  'electro',
        'FIGHT_PROP_GRASS_ADD_HURT': 'dendro',
        'FIGHT_PROP_ROCK_ADD_HURT':  'geo',
    }
    SUBSTAT_RE = re.compile(r'^([A-Z_]+)\+(-?[\d.]+)$')
    FILLER_ACTION = {
        'ganyu': 'aim',
        'neuvillette': 'charge',
    }

    # Traveler override handling
    traveler_override = traveler_override or {}

    # ---- Helper functions ----
    def _normalize(name):
        return re.sub(r"""[\s'".\-]""", "", str(name)).lower()

    def resolve_traveler(label, row, warnings):
        base = 'aether' if label.strip().lower() == 'aether' else 'lumine'
        if label in traveler_override:
            elem = traveler_override[label]
            warnings.append(f"{label}: using override element '{elem}'.")
            return f'{base}{elem}'
        for i in range(1, 6):
            main_type = row.get(f'Artifact {i} Main Stat Type')
            if pd.notna(main_type) and main_type in ELEMENT_DMG_PROP_TO_ELEMENT:
                elem = ELEMENT_DMG_PROP_TO_ELEMENT[main_type]
                warnings.append(f"{label}: auto-detected '{elem}' from goblet.")
                return f'{base}{elem}'
        for i in range(1, 6):
            for j in range(1, 5):
                sub = row.get(f'Artifact {i} Sub {j}')
                if pd.notna(sub):
                    for prop, elem in ELEMENT_DMG_PROP_TO_ELEMENT.items():
                        if str(sub).startswith(prop + '+'):
                            warnings.append(f"{label}: weak detection '{elem}' from substat.")
                            return f'{base}{elem}'
        warnings.append(f"{label}: defaulting to '{traveler_default}'.")
        return f'{base}{traveler_default}'

    def to_gcsim_name(csv_name, row, warnings):
        label = str(csv_name).strip()
        norm = _normalize(label)
        if norm in ('aether', 'lumine'):
            return resolve_traveler(label, row, warnings)
        if norm in CHARACTER_NAME_OVERRIDES:
            return CHARACTER_NAME_OVERRIDES[norm]
        if norm in GCSIM_CHARACTER_KEYS:
            return norm
        warnings.append(f"SKIPPED character '{label}': not in gcsim.")
        return None

    def to_gcsim_weapon(csv_weapon, char_label, warnings):
        norm = _normalize(csv_weapon)
        if norm in GCSIM_WEAPON_KEYS:
            return norm
        warnings.append(f"{char_label}: weapon '{csv_weapon}' not recognized.")
        return None

    def to_gcsim_set(csv_set, char_label, warnings):
        norm = _normalize(csv_set)
        if norm in GCSIM_SET_KEYS:
            return norm
        warnings.append(f"{char_label}: set '{csv_set}' not recognized.")
        return None

    def accumulate_artifact_stats(row, char_label, warnings):
        totals = {}
        unknown_types = set()

        def add(prop_type, raw_value):
            if prop_type not in FIGHT_PROP_TO_GCSIM:
                unknown_types.add(prop_type)
                return
            key, is_percent = FIGHT_PROP_TO_GCSIM[prop_type]
            v = raw_value / 100.0 if is_percent else raw_value
            totals[key] = totals.get(key, 0.0) + v

        for i in range(1, 6):
            main_type = row.get(f'Artifact {i} Main Stat Type')
            main_val = row.get(f'Artifact {i} Main Stat Value')
            if pd.notna(main_type) and pd.notna(main_val):
                add(str(main_type), float(main_val))
            for j in range(1, 5):
                sub = row.get(f'Artifact {i} Sub {j}')
                if pd.notna(sub):
                    m = SUBSTAT_RE.match(str(sub).strip())
                    if m:
                        add(m.group(1), float(m.group(2)))
                    else:
                        unknown_types.add(f"unparsable {sub!r}")
        if unknown_types:
            warnings.append(f"{char_label}: ignored unrecognized stat entries {sorted(unknown_types)}")
        return totals

    # ---- Build character configs ----
    def build_character_configs():
        df = pd.read_csv(csv_path, encoding='utf-8')
        df = df[df['Character Level'] >= min_character_level].copy()
        configs = {}
        skipped_unsupported = []
        warnings = []

        for _, row in df.iterrows():
            csv_name = row['Character Name']
            gcsim_name = to_gcsim_name(csv_name, row, warnings)
            if gcsim_name is None:
                skipped_unsupported.append(str(csv_name))
                continue

            char_label = f"{csv_name} -> {gcsim_name}"
            level = int(row['Character Level'])
            cons = int(row['Character Constellations']) if pd.notna(row.get('Character Constellations')) else 0
            na_raw = int(row['Talent NA Level']) if pd.notna(row.get('Talent NA Level')) else 1
            skill_raw = int(row['Talent Skill Level']) if pd.notna(row.get('Talent Skill Level')) else 1
            burst_raw = int(row['Talent Burst Level']) if pd.notna(row.get('Talent Burst Level')) else 1

            na, skill, burst = na_raw, skill_raw, burst_raw
            if cons >= 3 and skill > 10:
                skill = max(1, skill - 3)
                warnings.append(f"{char_label}: Skill boosted by C3, base {skill}.")
            if cons >= 5 and burst > 10:
                burst = max(1, burst - 3)
                warnings.append(f"{char_label}: Burst boosted by C5, base {burst}.")
            if na > 10:
                na = 10
                warnings.append(f"{char_label}: NA capped to 10.")
            na = max(1, min(10, na))
            skill = max(1, min(10, skill))
            burst = max(1, min(10, burst))

            lines = [
                f"{gcsim_name} char lvl={level}/90 cons={cons} talent={na},{skill},{burst} "
                f"+params=[start_energy=100];"
            ]
            weapon_name = row.get('Weapon Name')
            if pd.notna(weapon_name):
                weapon_key = to_gcsim_weapon(weapon_name, char_label, warnings)
                if weapon_key:
                    wlvl = int(row['Weapon Level']) if pd.notna(row.get('Weapon Level')) else 90
                    refine = int(row['Weapon Refinement']) if pd.notna(row.get('Weapon Refinement')) else 1
                    lines.append(f'{gcsim_name} add weapon="{weapon_key}" refine={refine} lvl={wlvl}/90;')
            set_counts = {}
            for i in range(1, 6):
                set_name = row.get(f'Artifact {i} Set')
                if pd.notna(set_name):
                    set_counts[set_name] = set_counts.get(set_name, 0) + 1
            for set_name, count in set_counts.items():
                if count < 2:
                    continue
                set_key = to_gcsim_set(set_name, char_label, warnings)
                if set_key:
                    lines.append(f'{gcsim_name} add set="{set_key}" count={count};')
            stat_totals = accumulate_artifact_stats(row, char_label, warnings)
            if stat_totals:
                stat_parts = [f'{k}={v:.4f}' for k, v in sorted(stat_totals.items())]
                lines.append(f'{gcsim_name} add stats {" ".join(stat_parts)};')
            if gcsim_name in configs:
                warnings.append(f"'{csv_name}' overwriting {gcsim_name}.")
            configs[gcsim_name] = '\n'.join(lines)

        print(f"Loaded {len(configs)} usable character(s): {sorted(configs.keys())}")
        if skipped_unsupported:
            print(f"\nSkipped {len(skipped_unsupported)} character(s) gcsim doesn't support yet:")
            for name in skipped_unsupported:
                print(f"  - {name}")
        if warnings:
            print(f"\n{len(warnings)} note(s)/warning(s):")
            for w in warnings:
                print(f"  - {w}")
        print()
        return configs

    char_configs = build_character_configs()
    all_chars = set(char_configs.keys())
    if lock_chars is None:
        lock_chars = []
    # Validate locked characters exist
    missing = [c for c in lock_chars if c not in all_chars]
    if missing:
        raise ValueError(f"Locked characters not in roster: {missing}")

    if len(all_chars) < 4:
        raise RuntimeError("Need at least 4 characters.")

    # ---- Rotation presets ----
    def preset_standard(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler};\n'
        return active, body

    def preset_support_first(team):
        main_dps = team[-1]
        others = team[:-1]
        active = team[0]
        body = ""
        for name in others:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        body += f'  if .{main_dps}.burst.ready {{ {main_dps} burst; }}\n'
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler};\n'
        return active, body

    def preset_quickswap(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler};\n'
        return active, body

    def preset_raiden_hyper(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        body += f'  if .{active}.name == "raiden" {{ {active} attack:15; }}\n'
        body += f'  else {{ {active} attack; }}\n'
        return active, body

    def preset_ganyu_aimed(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        body += f'  if .{active}.name == "ganyu" {{ {active} aim; }}\n'
        body += f'  else {{ {active} attack; }}\n'
        return active, body

    def preset_wriothesley_charge(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        body += f'  if .{active}.name == "wriothesley" {{ {active} charge; }}\n'
        body += f'  else {{ {active} attack; }}\n'
        return active, body

    def preset_universal_filler_heavy(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler},{filler},{filler};\n'
        return active, body

    def preset_burst_skill_weave(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler};\n'
        return active, body

    def preset_national(team):
        main_dps = team[-1]
        others = team[:-1]
        active = team[0]
        body = ""
        for name in others:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        body += f'  if .{main_dps}.burst.ready {{ {main_dps} burst; }}\n'
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler};\n'
        return active, body

    def preset_hyperbloom_driver(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler}:6;\n'   # 6 normal attacks
        return active, body

    def preset_battery_first(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler};\n'
        return active, body

    def preset_melt_ganyu_reverse(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        body += f'  if .{active}.name == "ganyu" {{ {active} aim; }}\n'
        body += f'  else {{ {active} attack; }}\n'
        return active, body

    def preset_tanky(team):
        active = team[0]
        body = ""
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        filler = FILLER_ACTION.get(active, 'attack')
        body += f'  {active} {filler},{filler};\n'   # 2 fillers
        return active, body

    def preset_kinich_skill(team):
        """Kinich on-field: skill, then use special attack chain."""
        active = team[0]
        body = ""
        # Use all bursts first (supports)
        for name in team:
            body += f'  if .{name}.burst.ready {{ {name} burst; }}\n'
        # Then skills
        for name in team:
            body += f'  if .{name}.skill.ready {{ {name} skill; }}\n'
        # If Kinich is active, do his enhanced string (attack:4 or charge based on his kit)
        body += f'  if .{active}.name == "kinich" {{ {active} attack:4; }}\n'
        body += f'  else {{ {active} attack; }}\n'
        return active, body



    ROTATION_PRESETS = [
    ("standard", preset_standard),
    ("support_first", preset_support_first),
    ("quickswap", preset_quickswap),
    ("raiden_hyper", preset_raiden_hyper),
    ("ganyu_aimed", preset_ganyu_aimed),
    ("wriothesley_charge", preset_wriothesley_charge),
    ("universal_heavy_filler", preset_universal_filler_heavy),
    ("burst_skill_weave", preset_burst_skill_weave),
    ("national", preset_national),
    ("hyperbloom_driver", preset_hyperbloom_driver),
    ("melt_ganyu_reverse", preset_melt_ganyu_reverse),
    ("battery_first", preset_battery_first),
    ("tanky", preset_tanky),
    ("kinich_skill", preset_kinich_skill),
    ]

    # ---- Build full config ----
    def build_full_config(team):
        preset_id, start_idx, *chars = team
        preset_func = ROTATION_PRESETS[preset_id][1]
        _, rotation_body = preset_func(chars)
        active = chars[start_idx]
        config = f"""
options iteration={sim_iterations} duration={sim_duration} swap_delay=4;
target lvl=100 resist=0.1 particle_threshold=250000 particle_drop_count=1;

"""
        for name in chars:
            config += char_configs[name] + '\n'
        config += f'active {active};\n\n'
        config += 'while 1 {\n'
        config += rotation_body
        config += '}\n'
        return config

    # ---- Run gcsim and return objectives dict ----
    def run_gcsim(config_str):
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
            f.write(config_str)
            config_path = f.name
        output_path = config_path + '.json'
        try:
            result = subprocess.run(
                [gcsim_bin, '-c', config_path, '-out', output_path],
                capture_output=True, text=True, timeout=120
            )
            if result.returncode != 0:
                return {'dps': 0.0, 'max_hit': 0.0, 'sd': 0.0}
            with open(output_path, 'r') as jf:
                data = json.load(jf)
            stats = data['statistics']
            dps = stats['dps']['mean']
            sd = stats['dps']['sd']

            # Extract max single hit from damage buckets
            max_hit = 0.0
            buckets_info = stats.get('damage_buckets')
            if buckets_info:
                bucket_list = buckets_info.get('buckets', [])
                for bucket in bucket_list:
                    if isinstance(bucket, dict):
                        max_hit = max(max_hit, bucket.get('max', 0.0))

            # Fallback to total_damage if buckets missing
            if max_hit == 0.0:
                max_hit = stats.get('total_damage', 0.0)

            return {'dps': dps, 'max_hit': max_hit, 'sd': sd}
        except Exception as e:
            import sys
            print(f"Exception in run_gcsim: {e}", file=sys.stderr)
            return {'dps': 0.0, 'max_hit': 0.0, 'sd': 0.0}
        finally:
            for f in [config_path, output_path]:
                if os.path.exists(f):
                    os.unlink(f)

    # ---- EA functions ----
    def random_team():
        preset = random.randrange(len(ROTATION_PRESETS))
        start_idx = random.randrange(4)
        # Always include locked characters first (order shuffled)
        locked = list(lock_chars)
        random.shuffle(locked)
        needed = 4 - len(locked)
        others = random.sample(list(all_chars - set(locked)), needed)
        chars = locked + others
        random.shuffle(chars)   # mix order so EA can evolve best sequence
        return (preset, start_idx) + tuple(chars)

    def fitness(team):
        config = build_full_config(team)
        return run_gcsim(config)

    def tournament_selection(population, scores):
        selected = random.sample(list(zip(population, scores)), tournament_size)
        selected.sort(key=lambda x: x[1]['dps'], reverse=True)
        return selected[0][0]

    def crossover(p1, p2):
        # inherit preset & start_idx
        child = [
            p1[0] if random.random() < 0.5 else p2[0],
            p1[1] if random.random() < 0.5 else p2[1],
        ]
        # crossover the 4 character slots
        raw_chars = []
        for i in range(2, len(p1)):
            raw_chars.append(p1[i] if random.random() < 0.5 else p2[i])

        # ---- enforce locked characters exactly once ----
        # 1. remove any extra locked characters (keep first occurrence)
        first_occurrence = {}
        for i, c in enumerate(raw_chars):
            if c in lock_chars and c not in first_occurrence:
                first_occurrence[c] = i

        # 2. fill missing locked characters
        present_locked = set(first_occurrence.keys())
        missing_locked = [lc for lc in lock_chars if lc not in present_locked]
        # indices of slots that are NOT a locked character
        free_slots = [i for i, c in enumerate(raw_chars) if c not in lock_chars]
        random.shuffle(free_slots)

        for lc in missing_locked:
            if free_slots:
                idx = free_slots.pop()
                raw_chars[idx] = lc
            else:
                # fallback: replace a random slot (shouldn’t happen if len(lock_chars) ≤ 4)
                idx = random.randrange(4)
                while raw_chars[idx] in lock_chars:
                    idx = random.randrange(4)
                raw_chars[idx] = lc

        # 3. fill remaining slots with unique non‑locked characters
        current_chars = set(raw_chars)
        needed = 4 - len(current_chars)
        if needed > 0:
            available = list(all_chars - current_chars)
            random.shuffle(available)
            for i in range(4):
                if raw_chars[i] in lock_chars and raw_chars.count(raw_chars[i]) > 1:
                    # replace duplicate locked (edge case)
                    raw_chars[i] = available.pop()
                elif raw_chars[i] not in lock_chars and current_chars:
                    # already fine
                    pass
            # a simpler final pass: just ensure no duplicates
            seen = set()
            for i in range(4):
                if raw_chars[i] in seen:
                    if available:
                        raw_chars[i] = available.pop()
                    else:
                        # fallback random
                        pool = list(all_chars - set(raw_chars))
                        if pool:
                            raw_chars[i] = random.choice(pool)
                seen.add(raw_chars[i])

        child.extend(raw_chars)
        return tuple(child)

    def mutate(team):
        if random.random() < mutation_rate:
            team_list = list(team)
            r = random.random()
            if r < 0.1:
                team_list[0] = random.randrange(len(ROTATION_PRESETS))
            elif r < 0.2:
                team_list[1] = random.randrange(4)
            else:
                # Only mutate a non‑locked character
                mutable_indices = [i for i in range(2, len(team_list))
                                  if team_list[i] not in lock_chars]
                if mutable_indices:
                    idx = random.choice(mutable_indices)
                    available = [c for c in all_chars
                                if c not in team_list[2:]]
                    if available:
                        team_list[idx] = random.choice(available)
            return tuple(team_list)
        return team

    def dominates(a, b):
        better = False
        if a['dps'] > b['dps']: better = True
        elif a['dps'] < b['dps']: return False
        if a['max_hit'] > b['max_hit']: better = True
        elif a['max_hit'] < b['max_hit']: return False
        if a['sd'] < b['sd']: better = True
        elif a['sd'] > b['sd']: return False
        return better

    # ---- CREATE OUTPUT FOLDER (per run) ----
    root_output = "optimizer_results"
    os.makedirs(root_output, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(root_output, f"{save_prefix}_{timestamp}")
    os.makedirs(run_dir, exist_ok=True)
    print(f"Output folder: {run_dir}")

    def save_file(filename, content):
        filepath = os.path.join(run_dir, filename)
        with open(filepath, 'w') as f:
            f.write(content)
        return filepath

    # ---- Main EA loop ----
    population = [random_team() for _ in range(population_size)]
    best_overall = (None, {'dps': 0.0, 'max_hit': 0.0, 'sd': 0.0})
    pareto_archive = []   # only used if pareto=True

    mode_str = "Pareto multi‑objective" if pareto else "pure DPS"
    print(f"Starting EA: pop={population_size}, gen={generations}, duration={sim_duration}s, iterations={sim_iterations}, mode={mode_str}")

    # Quick debug
    debug_team = (0, 0) + tuple(list(all_chars)[:4])
    test_res = fitness(debug_team)
    print(f"Debug team DPS: {test_res['dps']:.0f}, max_hit: {test_res['max_hit']:.0f}")

    try:
        for gen in range(generations):
            print(f"\nGeneration {gen+1}/{generations}")
            scores = [fitness(ind) for ind in population]
            unique_teams = set(population)
            print(f"Unique individuals: {len(unique_teams)}/{population_size}")

            # Update Pareto archive only if enabled
            if pareto:
                for team, obj in zip(population, scores):
                    dominated = False
                    for _, aobj in pareto_archive:
                        if dominates(aobj, obj):
                            dominated = True
                            break
                    if not dominated:
                        pareto_archive = [(t, o) for t, o in pareto_archive if not dominates(obj, o)]
                        pareto_archive.append((team, obj))

            fitness_vals = [s['dps'] for s in scores]
            top_indices = sorted(range(len(fitness_vals)), key=lambda i: fitness_vals[i], reverse=True)[:5]
            print("Top 5 this gen:")
            for idx in top_indices:
                preset_id, start_idx, *chars = population[idx]
                preset_name = ROTATION_PRESETS[preset_id][0]
                obj = scores[idx]
                print(f"  [{preset_name}] start:{chars[start_idx]} team:{chars} DPS:{obj['dps']:.0f} MaxHit:{obj['max_hit']:.0f}")

            best_idx = fitness_vals.index(max(fitness_vals))
            best_team = population[best_idx]
            best_obj = scores[best_idx]
            if best_obj['dps'] > best_overall[1]['dps']:
                best_overall = (best_team, best_obj)

            print(f"Gen best: {best_team} DPS:{best_obj['dps']:.0f} MaxHit:{best_obj['max_hit']:.0f}")
            print(f"Overall best: {best_overall[0]} DPS:{best_overall[1]['dps']:.0f} MaxHit:{best_overall[1]['max_hit']:.0f}")

            elite_count = max(1, int(population_size * 0.2))
            combined = sorted(zip(population, fitness_vals), key=lambda x: x[1], reverse=True)
            elites = [ind for ind, _ in combined[:elite_count]]
            new_pop = elites.copy()
            while len(new_pop) < population_size:
                p1 = tournament_selection(population, scores)
                p2 = tournament_selection(population, scores)
                if random.random() < crossover_rate:
                    child = crossover(p1, p2)
                else:
                    child = p1
                child = mutate(child)
                new_pop.append(child)
            population = new_pop

    except KeyboardInterrupt:
        print("\nInterrupted! Saving best so far.")
        if best_overall[0] is not None:
            config = build_full_config(best_overall[0])
            save_file("best_team_interrupted.txt", config)
        if pareto and pareto_archive:
            pareto_sorted = sorted(pareto_archive, key=lambda x: x[1]['dps'], reverse=True)[:10]
            for i, (t, obj) in enumerate(pareto_sorted):
                config = build_full_config(t)
                save_file(f"pareto_interrupted_{i+1}.txt", config)
        raise

    # Final output
    print("\n===== FINAL BEST TEAM (by DPS) =====")
    best_team_tuple = best_overall[0]
    best_obj = best_overall[1]
    best_config = build_full_config(best_team_tuple)
    print(f"DPS: {best_obj['dps']:.0f}  MaxHit: {best_obj['max_hit']:.0f}")
    print(best_config)
    save_file("best_team.txt", best_config)

    if pareto and pareto_archive:
        print("\n===== PARETO FRONT (top 10) =====")
        pareto_archive.sort(key=lambda x: x[1]['dps'], reverse=True)
        top_10_pareto = pareto_archive[:10]
        for i, (team, obj) in enumerate(top_10_pareto):
            preset_id, start_idx, *chars = team
            preset_name = ROTATION_PRESETS[preset_id][0]
            print(f"#{i+1} [{preset_name}] start:{chars[start_idx]} {chars}")
            print(f"   DPS:{obj['dps']:.0f}  MaxHit:{obj['max_hit']:.0f}  SD:{obj['sd']:.0f}")
            config = build_full_config(team)
            save_file(f"pareto_{i+1}.txt", config)
        # Save a JSON summary of the Pareto front
        summary = []
        for team, obj in top_10_pareto:
            preset_id, start_idx, *chars = team
            summary.append({
                'preset': ROTATION_PRESETS[preset_id][0],
                'start_char': chars[start_idx],
                'team': chars,
                'dps': obj['dps'],
                'max_hit': obj['max_hit'],
                'sd': obj['sd']
            })
        with open(os.path.join(run_dir, 'pareto_summary.json'), 'w') as f:
            json.dump(summary, f, indent=2)
    else:
        print("\n(Pareto archive disabled – only the single best DPS team is shown)")



In [ ]:
# ---- Call the function with your desired settings ----
if __name__ == "__main__":
    # Example 1: multi‑objective (Pareto) – you get best DPS + nuke + consistent teams
    run_optimizer(
        lock_chars=["neuvillette","furina","jean","fischl"],
        csv_path="genshin_data_export.csv",
        gcsim_bin="./gcsim",
        sim_duration=90,
        sim_iterations=10,
        population_size=50,
        generations=50,
        mutation_rate=0.4,
        save_prefix="general",
        pareto=True          # <-- this enables the Pareto front
    )

    # Example 2: pure DPS – only the team with the highest average DPS matters
    # run_optimizer(
    #     lock_chars=["neuvillette", "furina"],
    #     csv_path="genshin_data_export.csv",
    #     gcsim_bin="./gcsim",
    #     sim_duration=90,
    #     generations=50,
    #     population_size=150,
    #     save_prefix="pure_dps_test",
    #     pareto=False
    # )